# 💡 SQL Track – Lösungen

Dieses Notebook enthält die Lösungen aller **🏋️ Du bist dran**-Übungen des SQL Tracks, in der Reihenfolge der Labs.

> **❗ Wichtig:** Versuche dich zuerst selbst an jeder Übung – aus einer Abfrage, an der du geknobelt hast, lernst du viel mehr als aus einer kopierten. Schau hier nach, um dein Ergebnis zu prüfen oder wenn du nicht weiterkommst.

Jede Lösung ist eine SQL-Zelle, die du direkt hier ausführen kannst. Verbinde das Notebook mit **Serverless**-Compute oder einem **SQL warehouse** und setze zuerst den Standardkatalog (ersetze ihn, falls du in Lab 0 einen anderen Katalog verwendet hast):

In [ ]:
USE CATALOG sunny_bay_germany;

> **📝 Hinweis:** Wenn du in Lab 0 ein **Präfix** verwendet hast, beginnen alle Tabellennamen damit (z. B. `silver.jd_customer` statt `silver.customer`).

## SQL 1 – Daten filtern, aggregieren und verknüpfen

### SQL 1 › Schritt 2 › Aufgabe 4: Zeilen filtern mit WHERE

Finde alle Bestellungen in `silver.sales_order` aus dem **Berliner Café** (`store_id = 2`), die in den **letzten 7 Tagen** mit **Girocard** bezahlt wurden. Zeige Bestell-ID, Zeitstempel und Zahlungsart, die neuesten zuerst.

In [ ]:
SELECT order_id, order_ts, payment_method
FROM silver.sales_order
WHERE store_id = 2
  AND payment_method = 'Girocard'
  AND order_date >= current_date() - INTERVAL 7 DAYS
ORDER BY order_ts DESC;

### SQL 1 › Schritt 3 › Aufgabe 4: Zusammenfassen mit GROUP BY und HAVING

Der CRM-Export `bronze.crm_customers` sollte eine Zeile pro Kunde enthalten – aber stimmt das? Finde alle `customer_id`s, die **mehr als einmal** vorkommen, und zeige, wie viele Zeilen jede davon hat.

In [ ]:
SELECT customer_id, COUNT(*) AS versions
FROM bronze.crm_customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY versions DESC, customer_id;

Das CRM liefert **Historienzeilen**: Wenn ein Kunde seine E-Mail-Adresse ändert oder umzieht, kommt eine neue Version mit einem neueren `updated_at` hinzu. Wie du mit `QUALIFY` nur die neueste Version behältst, lernst du in SQL 3 – und in Schritt 6 siehst du, was passiert, wenn du das vergisst.

### SQL 1 › Schritt 5 › Aufgabe 4: INNER JOIN – nur passende Zeilen

Berechne den **Bruttoumsatz pro Produktname** (`quantity * unit_price_eur - discount_eur`) und zeige die 10 umsatzstärksten Produkte.

In [ ]:
SELECT
    p.product_name,
    ROUND(SUM(l.quantity * l.unit_price_eur - l.discount_eur), 2) AS gross_revenue_eur
FROM silver.sales_order_line AS l
INNER JOIN silver.product AS p
    ON l.product_id = p.product_id
GROUP BY p.product_name
ORDER BY gross_revenue_eur DESC
LIMIT 10;

### SQL 1 › Schritt 6 › Aufgabe 3: LEFT JOIN – alles von links behalten

Wie viele Kunden in `silver.customer` hatten **noch nie** eine Kundenkarte? Verwende einen `LEFT JOIN` und suche nach den `NULL`s.

In [ ]:
SELECT COUNT(*) AS customers_without_card
FROM silver.customer AS c
LEFT JOIN silver.loyalty_card AS lc
    ON c.customer_id = lc.customer_id
WHERE lc.card_no IS NULL;

Dieses Muster „LEFT JOIN + IS NULL“ ist der klassische Weg, Zeilen **ohne** Treffer zu finden. In Schritt 9 lernst du eine kürzere Schreibweise dafür kennen: den Anti Join.

### SQL 1 › Schritt 7 › Aufgabe 3: FULL OUTER JOIN – zwei Systeme vergleichen

Warum vergleichen wir `LOWER(TRIM(crm.email))` und nicht einfach `crm.email`? Zähle die übereinstimmenden Zeilen (`INNER JOIN`) einmal **mit** und einmal **ohne** `LOWER(TRIM(...))`. Sieh dir dann einige CRM-E-Mails an, die nicht schon kleingeschrieben sind.

In [ ]:
SELECT
    (SELECT COUNT(*)
     FROM bronze.crm_customers AS crm
     INNER JOIN bronze.web_accounts AS web ON crm.email = web.email)              AS raw_matches,
    (SELECT COUNT(*)
     FROM bronze.crm_customers AS crm
     INNER JOIN bronze.web_accounts AS web ON LOWER(TRIM(crm.email)) = web.email) AS normalized_matches;

SELECT customer_id, email
FROM bronze.crm_customers
WHERE email <> LOWER(TRIM(email))
LIMIT 20;

Das CRM speichert E-Mails genau so, wie das Café-Personal sie eingetippt hat – mit Großbuchstaben und Leerzeichen am Anfang oder Ende. Der Onlineshop speichert sie kleingeschrieben. **Join-Schlüssel müssen normalisiert werden**, bevor sie übereinstimmen können. Die Silver-Tabelle `silver.customer` macht genau das (und entfernt außerdem die CRM-Historienzeilen), bevor sie beide Systeme mit einem Full Outer Join zusammenführt.

### SQL 1 › Schritt 8 › Aufgabe 4: LEFT SEMI JOIN – „hat mindestens einen Treffer“

Welche Kundenkarten wurden in den **letzten 30 Tagen** für mindestens einen Einkauf im Café verwendet? Zeige Kartennummer, Kunde und Stufe.

In [ ]:
SELECT lc.card_no, lc.customer_id, lc.loyalty_tier
FROM silver.loyalty_card AS lc
LEFT SEMI JOIN silver.sales_order AS o
    ON  lc.card_no = o.loyalty_card_no
    AND o.order_date >= current_date() - INTERVAL 30 DAYS
ORDER BY lc.card_no;

### SQL 1 › Schritt 9 › Aufgabe 3: LEFT ANTI JOIN – „hat keinen Treffer“

Welche Produkte aus den Produktstammdaten wurden **noch nie verkauft**? Schreibe die Abfrage einmal mit `LEFT ANTI JOIN` und einmal mit `NOT EXISTS`. Vergleiche dein Ergebnis dann mit der Spalte `has_sales` von `gold.dim_product`.

In [ ]:
SELECT p.product_id, p.product_name, p.product_status, p.launched_on
FROM silver.product AS p
LEFT ANTI JOIN silver.sales_order_line AS l
    ON p.product_id = l.product_id;

SELECT p.product_id, p.product_name, p.product_status, p.launched_on
FROM silver.product AS p
WHERE NOT EXISTS (
    SELECT 1 FROM silver.sales_order_line AS l WHERE l.product_id = p.product_id
);

SELECT product_key, product_name, has_sales
FROM gold.dim_product
WHERE NOT has_sales;

Du solltest drei Produkte finden: Eines wurde schon vor dem Verkaufszeitraum aus dem Sortiment genommen, zwei kommen demnächst auf den Markt.

### SQL 1 › Schritt 10 › Aufgabe 3: CROSS JOIN – jede Kombination

Wie viele Zeilen hat das Raster **vor** dem Anti Join? Zähle die Filialen, die Tage des Verkaufszeitraums und die Zeilen des Cross Joins und prüfe, ob Zeilen = Filialen × Tage gilt.

In [ ]:
SELECT
    COUNT(DISTINCT s.store_id) AS stores,
    COUNT(DISTINCT d.date)     AS days,
    COUNT(*)                   AS grid_rows
FROM silver.store AS s
CROSS JOIN gold.dim_date AS d
WHERE d.date BETWEEN (SELECT MIN(order_date) FROM silver.sales_order)
                 AND (SELECT MAX(order_date) FROM silver.sales_order);

## SQL 2 – Daten kombinieren und analysieren

### SQL 2 › Schritt 1 › Aufgabe 3: Ergebnismengen stapeln mit UNION ALL und UNION

Erstelle eine gemeinsame Liste aller **Städte**, in denen Sunny Bay Kunden hat – aus dem CRM (`bronze.crm_customers.city`) und aus dem Onlineshop (`bronze.web_accounts.city`). Jede Stadt soll nur einmal vorkommen, alphabetisch sortiert.

In [ ]:
SELECT city FROM bronze.crm_customers
UNION
SELECT city FROM bronze.web_accounts
ORDER BY city;

`UNION` (ohne `ALL`) entfernt die Duplikate. Beachte, dass `ORDER BY` am Ende das **kombinierte** Ergebnis sortiert.

### SQL 2 › Schritt 2 › Aufgabe 4: Listen vergleichen mit MINUS und INTERSECT

Welche E-Mail-Adressen gibt es **sowohl** im CRM (`bronze.crm_customers`) **als auch** im Onlineshop (`bronze.web_accounts`)? Denk an die Lektion aus SQL 1: Normalisiere die CRM-E-Mails mit `LOWER(TRIM(...))`. Zähle sie anschließend.

In [ ]:
SELECT COUNT(*) AS emails_in_both_systems
FROM (
    SELECT LOWER(TRIM(email)) AS email FROM bronze.crm_customers
    INTERSECT
    SELECT email FROM bronze.web_accounts
) AS both_systems;

Das sind die **Omnichannel**-Kunden, die in den Cafés *und* online kaufen. `silver.customer` führt jeden von ihnen zu einem einzigen Kunden zusammen.

### SQL 2 › Schritt 3 › Aufgabe 3: Abfragen strukturieren mit WITH … AS

Die verschachtelte Abfrage im Lab findet die Filialen, deren Anzahl an **Samstags**-Bestellungen über dem Durchschnitt aller Filialen liegt. Sie funktioniert, ist aber schwer zu lesen. Schreibe sie mit zwei CTEs um: `saturday_orders` (Bestellungen pro Filiale) und `average` (der Durchschnitt dieser Zahlen).

In [ ]:
WITH saturday_orders AS (
    SELECT s.store_name, COUNT(*) AS saturday_orders
    FROM silver.sales_order AS o
    INNER JOIN silver.store AS s ON o.store_id = s.store_id
    WHERE DAYOFWEEK(o.order_date) = 7      -- 1 = Sonntag ... 7 = Samstag
    GROUP BY s.store_name
),
average AS (
    SELECT AVG(saturday_orders) AS avg_saturday_orders
    FROM saturday_orders
)
SELECT so.store_name, so.saturday_orders
FROM saturday_orders AS so
CROSS JOIN average AS a
WHERE so.saturday_orders > a.avg_saturday_orders
ORDER BY so.saturday_orders DESC;

Die CTE `saturday_orders` wird zweimal verwendet – einmal für den Durchschnitt und einmal für das Endergebnis. Der `CROSS JOIN` mit einer einzeiligen CTE ist ein gängiger Trick, um einen einzelnen Wert an jede Zeile anzuhängen.

### SQL 2 › Schritt 4 › Aufgabe 5: Analysieren mit Window-Funktionen und PARTITION BY

Finde für jede Kundenkarte die **erste Café-Bestellung** (die Bestellung mit dem frühesten `order_ts`). Nummeriere die Bestellungen pro Karte in einer CTE mit `ROW_NUMBER()` und behalte nur Nummer 1. Zeige Kartennummer, Bestell-ID und Zeitstempel.

In [ ]:
WITH numbered_orders AS (
    SELECT
        loyalty_card_no,
        order_id,
        order_ts,
        ROW_NUMBER() OVER (PARTITION BY loyalty_card_no ORDER BY order_ts) AS order_no
    FROM silver.sales_order
    WHERE loyalty_card_no IS NOT NULL
)
SELECT loyalty_card_no, order_id, order_ts
FROM numbered_orders
WHERE order_no = 1
ORDER BY order_ts;

Du kannst nicht direkt `WHERE ROW_NUMBER() OVER (…) = 1` schreiben, weil `WHERE` **vor** den Window-Funktionen ausgewertet wird. Deshalb brauchen wir die CTE – oder die `QUALIFY`-Klausel, die du in SQL 3 kennenlernst.

### SQL 2 › Schritt 5 › Aufgabe 4: Zwischensummen mit ROLLUP, CUBE und GROUPING()

Zeige den Bruttoumsatz pro **Filiale** und **Produktkategorie** (aus `gold.dim_store` und `gold.dim_product`), mit einer Zwischensumme pro Filiale und einer Gesamtsumme. Beschrifte die Zwischensummenzeilen.

In [ ]:
SELECT
    s.store_name,
    p.product_category,
    SUM(f.gross_revenue_eur) AS revenue_eur,
    CASE
        WHEN GROUPING(s.store_name) = 1       THEN 'Grand total'
        WHEN GROUPING(p.product_category) = 1 THEN 'Store total'
        ELSE 'Category'
    END AS row_type
FROM gold.fact_coffee_sales AS f
INNER JOIN gold.dim_store   AS s ON f.store_key = s.store_key
INNER JOIN gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY ROLLUP (s.store_name, p.product_category)
ORDER BY s.store_name NULLS LAST, p.product_category NULLS LAST;

### SQL 2 › Schritt 6 › Aufgabe 3: Hierarchien durchlaufen mit WITH RECURSIVE

Baue den **Kategoriepfad** jedes Produkts auf, z. B. `All Products > Drink > Milk Coffee > Cappuccino`. Starte die Rekursion bei der Wurzelkategorie (`parent_category_id IS NULL`) von `silver.product_category`, laufe den Baum hinab und verknüpfe zum Schluss die Produkte mit ihrer Kategorie (`silver.product.category_id`).

In [ ]:
WITH RECURSIVE category_path (category_id, path, depth) AS (
    SELECT category_id, category_name, 1
    FROM silver.product_category
    WHERE parent_category_id IS NULL

    UNION ALL

    SELECT c.category_id, CONCAT(cp.path, ' > ', c.category_name), cp.depth + 1
    FROM silver.product_category AS c
    INNER JOIN category_path AS cp
        ON c.parent_category_id = cp.category_id
)
SELECT
    p.product_name,
    CONCAT(cp.path, ' > ', p.product_name) AS full_path,
    cp.depth + 1                           AS depth
FROM silver.product AS p
INNER JOIN category_path AS cp
    ON p.category_id = cp.category_id
ORDER BY full_path;

> **✨ Tipp:** Wirf einen Blick in `bundle/src/transformations/gold/dim_product.sql`: Der Gold Layer klappt den Kategoriebaum **ohne** Rekursion auf – er joint `product_category` zweimal (ein *Self-Join*). Das klappt, weil der Baum eine feste Tiefe von drei Ebenen hat. Verwende Rekursion, wenn die Tiefe variieren kann (wie bei einem Organigramm).

## SQL 3 – Funktionen für den Gold Layer

### SQL 3 › Schritt 1 › Aufgabe 4: Arrays auflösen mit EXPLODE

Wie viele **Einheiten pro SKU** wurden online bestellt? Arbeite direkt mit dem Roh-JSON in `bronze.web_orders` (ignoriere stornierte Bestellungen) und sortiere nach Einheiten.

In [ ]:
WITH web_items AS (
    SELECT EXPLODE(items) AS item
    FROM bronze.web_orders
    WHERE status <> 'cancelled'
)
SELECT item.sku, SUM(item.qty) AS units
FROM web_items
GROUP BY item.sku
ORDER BY units DESC;

### SQL 3 › Schritt 2 › Aufgabe 4: Die richtige Zeile pro Gruppe behalten mit QUALIFY

In SQL 2 hast du die Produkte pro Filiale mit einer CTE in eine Rangfolge gebracht. Zeige jetzt nur die **Top 3 Produkte pro Filiale** nach Bruttoumsatz – in einer einzigen Abfrage mit `QUALIFY` (verwende `gold.fact_coffee_sales`, `gold.dim_store` und `gold.dim_product`).

In [ ]:
SELECT
    s.store_name,
    p.product_name,
    SUM(f.gross_revenue_eur) AS revenue_eur
FROM gold.fact_coffee_sales AS f
INNER JOIN gold.dim_store   AS s ON f.store_key = s.store_key
INNER JOIN gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY s.store_name, p.product_name
QUALIFY RANK() OVER (PARTITION BY s.store_name ORDER BY SUM(f.gross_revenue_eur) DESC) <= 3
ORDER BY store_name, revenue_eur DESC;

`QUALIFY` wird **nach** `GROUP BY` ausgewertet, daher kann die Window-Funktion sogar aggregierte Werte in eine Rangfolge bringen.

### SQL 3 › Schritt 3 › Aufgabe 4: Fehlende Werte ersetzen mit COALESCE

Online-Bestellungen haben eine Lieferstadt (`ship_to_city`), Café-Bestellungen nicht. Zähle die Bestellungen pro **„Bestellstadt“**: bei Online-Bestellungen die Lieferstadt, sonst die Stadt der Filiale (verknüpfe dazu `silver.store`).

In [ ]:
SELECT
    COALESCE(o.ship_to_city, s.city) AS order_city,
    COUNT(*)                         AS orders
FROM silver.sales_order AS o
INNER JOIN silver.store AS s
    ON o.store_id = s.store_id
GROUP BY COALESCE(o.ship_to_city, s.city)
ORDER BY orders DESC;

### SQL 3 › Schritt 4 › Aufgabe 3: Werte ableiten mit CASE und IFF

Zähle die Kunden in `silver.customer` pro **Altersgruppe**: `Under 30`, `30–49`, `50 and older` und `Unknown` (reine Onlineshop-Kunden haben kein Geburtsdatum). Berechne das Alter als `YEAR(current_date()) - YEAR(birth_date)`.

In [ ]:
SELECT
    CASE
        WHEN birth_date IS NULL                              THEN 'Unknown'
        WHEN YEAR(current_date()) - YEAR(birth_date) < 30    THEN 'Under 30'
        WHEN YEAR(current_date()) - YEAR(birth_date) < 50    THEN '30–49'
        ELSE '50 and older'
    END      AS age_group,
    COUNT(*) AS customers
FROM silver.customer
GROUP BY 1
ORDER BY 1;

Beachte, dass die `NULL`-Prüfung **zuerst** kommt: Ein Vergleich mit `NULL` ist nie wahr – ohne die Prüfung würden diese Kunden also im `ELSE`-Zweig landen.

### SQL 3 › Schritt 5 › Aufgabe 4: Codes übersetzen mit DECODE

Mach die rohe ERP-Produktliste in `bronze.erp_products` lesbar: Übersetze `channel_code` (`S` = „Café only“, `O` = „Online only“, `B` = „Café and online“), `status_code` (`A` = „Active“, `D` = „Discontinued“) und `tax_class` (`S` = 0.19, `R` = 0.07 als Mehrwertsteuersatz).

In [ ]:
SELECT
    sku,
    product_name,
    DECODE(channel_code, 'S', 'Café only', 'O', 'Online only', 'B', 'Café and online', 'Unknown') AS sales_channels,
    DECODE(status_code, 'A', 'Active', 'D', 'Discontinued', 'Unknown')                            AS product_status,
    DECODE(tax_class, 'S', 0.19, 'R', 0.07)                                                       AS vat_rate
FROM bronze.erp_products
ORDER BY sku;

Vergleiche dein Ergebnis mit `silver.product`, das dieselben Spalten übersetzt.

### SQL 3 › Schritt 6 › Aufgabe 4: Den Wert am Maximum wählen mit MAX_BY

Finde den **besten Tag** jeder Filiale: das Datum mit dem höchsten Bruttoumsatz und den Umsatz an diesem Tag.

In [ ]:
WITH daily AS (
    SELECT s.store_name, d.date, SUM(f.gross_revenue_eur) AS revenue_eur
    FROM gold.fact_coffee_sales AS f
    INNER JOIN gold.dim_store AS s ON f.store_key = s.store_key
    INNER JOIN gold.dim_date  AS d ON f.date_key = d.date_key
    GROUP BY s.store_name, d.date
)
SELECT
    store_name,
    MAX_BY(date, revenue_eur) AS best_day,
    MAX(revenue_eur)          AS best_day_revenue_eur
FROM daily
GROUP BY store_name
ORDER BY store_name;

### SQL 3 › Schritt 7 › Aufgabe 3: Bedingte Aggregate mit FILTER (WHERE …)

Zähle für jedes **Café** die **bar** bezahlten und die **bargeldlos** bezahlten Bestellungen (alle anderen Zahlungsarten) und berechne den Baranteil in Prozent.

In [ ]:
SELECT
    s.store_name,
    COUNT(*) FILTER (WHERE o.payment_method = 'Cash')                       AS cash_orders,
    COUNT(*) FILTER (WHERE o.payment_method <> 'Cash')                      AS cashless_orders,
    ROUND(100 * COUNT(*) FILTER (WHERE o.payment_method = 'Cash') / COUNT(*), 1) AS cash_share_pct
FROM silver.sales_order AS o
INNER JOIN silver.store AS s
    ON o.store_id = s.store_id
WHERE o.sales_channel = 'In-store'
GROUP BY s.store_name
ORDER BY cash_share_pct DESC;

### SQL 3 › Schritt 8 › Aufgabe 4: Mit Datumswerten rechnen – DATEADD und DATEDIFF

Neue Mitglieder des Treueprogramms erhalten in ihrem **ersten Mitgliedsjahr** einen Willkommensbonus. Berechne für jede **aktive** Karte das Datum, an dem das erste Jahr endet, und zeige, ob das Mitglied heute noch im ersten Jahr ist.

In [ ]:
SELECT
    card_no,
    customer_id,
    enrolled_on,
    CAST(DATEADD(YEAR, 1, enrolled_on) AS DATE)               AS first_year_ends_on,
    current_date() < CAST(DATEADD(YEAR, 1, enrolled_on) AS DATE) AS in_first_year
FROM silver.loyalty_card
WHERE is_active
ORDER BY enrolled_on DESC;

## 🎉 Geschafft

Zurück zu den Labs: **SQL 1 – Filtern, Aggregieren und Verknüpfen**, **SQL 2 – Daten kombinieren und analysieren** und **SQL 3 – Funktionen für den Gold Layer**.